In [55]:
import os

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model='Qwen/Qwen2.5-72B-Instruct',
    base_url=os.environ.get('OPENAI_BASE_URL'),
    api_key=os.environ.get('OPENAI_API_KEY'),
    temperature=0
)

In [56]:
from enum import Enum
from pydantic import BaseModel, Field


class Sentiment(str, Enum):
    HAPPY = 'happy'
    SAD = 'sad'
    NEUTRAL = 'neutral'
    UNKNOWN = 'unknown'


class Classifier(BaseModel):
    """
        定义一个Pydantic的数据模型，未来需要根据该类型，完成文本的分类
    """
    # 文本的情感倾向，预期为字符串类型
    sentiment: Sentiment = Field(default=Sentiment.UNKNOWN, description="文本的情感")

    # 文本的攻击性，预期为1到5的整数
    aggressiveness: int = Field(..., enum=[1, 2, 3, 4, 5], description="描述文本的攻击性，数字越大表示越攻击性")

    # 文本使用的语言，预期为字符串类型
    language: str = Field(..., enum=["spanish", "english", "french", "中文", "italian"], description="文本使用的语言")

C:\Users\14820\AppData\Local\Temp\ipykernel_6440\3467112139.py:20: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'enum'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  aggressiveness: int = Field(..., enum=[1, 2, 3, 4, 5], description="描述文本的攻击性，数字越大表示越攻击性")
C:\Users\14820\AppData\Local\Temp\ipykernel_6440\3467112139.py:23: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'enum'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  language: str = Field(..., enum=["spanish", "english", "french", "中文", "italian"], description="文本使用的语言")


In [57]:
from langchain_core.prompts import ChatPromptTemplate

tagging_prompt = ChatPromptTemplate.from_template("""
从以下段落中提取信息。
只提取'Classifier'类中提到的属性。
段落：
{input}
""")

chain = tagging_prompt | model.with_structured_output(Classifier)

In [58]:
chain.invoke({"input": "中国人民大学的王教授，师德败坏，做出的事情实在让我很生气!"})

Classifier(sentiment=<Sentiment.NEUTRAL: 'neutral'>, aggressiveness=1, language='中文')

In [59]:
chain.invoke({"input": "我非常开心，因为今天是星期五"})

Classifier(sentiment=<Sentiment.HAPPY: 'happy'>, aggressiveness=1, language='中文')

In [60]:
chain.invoke("fuck you!")

Classifier(sentiment=<Sentiment.NEUTRAL: 'neutral'>, aggressiveness=1, language='english')